# Build Your AI Clone

In this workshop you'll analyze your own WhatsApp chat with Python and build an AI that texts like you.

**How this notebook works**

- It's made of **cells**. Grey cells have code, white cells (like this one) have explanations.
- To run a code cell, click on it and press **Shift + Enter**. Run them in order, from top to bottom.
- Lines that start with `#` are **comments**: notes for humans that Python ignores.
- Cells marked with **Challenge** are for you to change and run again. You can't break anything!

## Setup

The hard parts of this workshop (reading the WhatsApp format, drawing charts, the chat window...) are in a separate file called `workshop.py`. This cell downloads it and uses **`import`** to bring all its tools into the notebook, so we can use them without writing them ourselves. That's how programmers work: we reuse code instead of writing everything from zero.

It also installs **Gradio**, a library to make little web apps. It takes about 30 seconds.

In [ ]:
!pip install -q gradio
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/MateoNeiraCastillo/coding-club-workshops/main/01-ai-clone/workshop.py", "workshop.py")
from workshop import *

print("Ready!")

## Step 1 · Get your chat

First, export one of your WhatsApp chats **without media**:

- **iPhone:** open the chat → tap the name at the top → **Export Chat** → **Without Media**
- **Android:** open the chat → ⋮ → **More** → **Export chat** → **Without media**

Send the file to your laptop. Both `.txt` and `.zip` files work.

Don't want to use your own chat? Type `y` when asked and you'll get **"Mentors Anonymous"**, a group chat between **Tony Stark, Yoda and Haymitch** (all the messages are original, written for this workshop).

**New concept: functions and variables.** `get_chat()` is a **function**: a piece of code with a name that does a job. This one asks you for the chat and **returns** its text. With `=` we save that text in a **variable** called `chat_text`, so we can use it later.

In [ ]:
chat_text = get_chat()

## Step 2 · Read the chat

**New concept: lists and dictionaries.** `read_chat` turns the text into a **list** of messages. A list is a group of things in order, written between `[ ]`.

Each message is a **dictionary**: a group of values, each one with a name (a **key**). Every message has three keys: `"date"`, `"author"` and `"text"`.

In [ ]:
messages = read_chat(chat_text)
show_messages(messages)

You get one item of a list with its position between `[ ]`. Python starts counting at **0**, so `messages[0]` is the first message. Then you get one value of a dictionary with its key: `first_message["author"]`.

Try changing the `0` to `1`, `2`... or to `-1` (the last message!) and run the cell again.

In [ ]:
first_message = messages[0]

print(first_message["author"])
print(first_message["text"])

## Step 3 · Who are you?

**New concept: loops and conditions.** A `for` **loop** repeats the same code for every item of a list. Here we go through every message and, **if** its author is not in our list yet, we add them with `.append()`.

Notice the spaces at the start of the lines: they tell Python which code is inside the loop and which is inside the `if`.

In [ ]:
authors = []

for message in messages:
    if message["author"] not in authors:
        authors.append(message["author"])

show_people(authors)

Now type your number. `input()` asks you something and `int()` turns your answer into a number.

We do `choice - 1` because the list starts counting at 0. Then we use another loop to keep only **your** messages.

The `f` before the quotes makes an **f-string**: whatever is inside `{ }` gets replaced by its value.

In [ ]:
choice = int(input("Which number is you? "))
my_name = authors[choice - 1]

my_messages = []
for message in messages:
    if message["author"] == my_name:
        my_messages.append(message)

card(f"Hi {my_name}!", f"You sent {len(my_messages)} messages in this chat.")

## Step 4 · Your WhatsApp Wrapped

**New concept: counting with `Counter`.** A `Counter` is a special dictionary for counting things. Every time we do `+= 1`, we add one to that person's count. Then `.most_common(10)` gives us the 10 biggest counts.

In [ ]:
message_counts = Counter()

for message in messages:
    message_counts[message["author"]] += 1

bar_chart(message_counts.most_common(10), "Who texts the most", color=RED)

Same idea, but now we count **your words**. There are two loops, one inside the other: for every message, and then for every word in that message. We skip boring words like "the" or "que" (they're in `STOP_WORDS`).

In [ ]:
word_counts = Counter()

for message in my_messages:
    for word in get_words(message["text"]):
        if word not in STOP_WORDS:
            word_counts[word] += 1

bar_chart(word_counts.most_common(10), "Your top 10 words")

### Challenge 1

In the cell above, change it to show your **top 5** words instead of 10, and change its color by adding `, color=RED` after the title (or any color like `"#00B894"`). Then run it again.

### Challenge 2

How many times do you say a word? Change `"hello"` to any word or name and run the cell.

In [ ]:
search = "hello"

times = 0
for message in my_messages:
    if search in message["text"].lower():
        times += 1

print(f'You wrote "{search}" in {times} messages')

## Step 5 · Pick the messages for your clone

First we save 5 **real moments** for the game at the end: a message someone sent you and what you really answered. Your clone will never see these replies, so the game is fair.

We go through the messages with `range()`, which gives us the positions 1, 2, 3... so we can look at each message (`messages[i]`) and the one before it (`messages[i - 1]`). Subtracting two dates gives the time between them.

A pair of values in `( )` is called a **tuple**.

In [ ]:
moments = []

for i in range(1, len(messages)):
    previous = messages[i - 1]
    reply = messages[i]
    minutes = (reply["date"] - previous["date"]).total_seconds() / 60

    if previous["author"] != my_name and reply["author"] == my_name and minutes < 60:
        if len(reply["text"].split()) >= 3:
            moments.append((previous["text"], reply["text"]))

game_moments = random.sample(moments, min(5, len(moments)))
print(f"Found {len(moments)} moments, saved {len(game_moments)} for the game.")

Now we collect the messages your clone can learn from: yours, with at least 3 words, and not the ones saved for the game. Then you choose how many (up to 1000): more messages usually make a better clone.

`random.sample` picks messages at random, and `min()` gives the smallest number, so we never ask for more messages than you have.

**These messages will be sent to Google's AI (Gemini).** On the free plan, Google may keep them and people at Google may read them.

In [ ]:
saved_replies = []
for friend_message, real_reply in game_moments:
    saved_replies.append(real_reply)

candidates = []
for message in my_messages:
    text = message["text"]
    if len(text.split()) >= 3 and text not in saved_replies:
        candidates.append(text)

print(f"You have {len(candidates)} messages with 3 or more words.")

amount = int(input("How many do you want your clone to learn from? (up to 1000): "))
amount = min(amount, 1000, len(candidates))
examples = random.sample(candidates, amount)

To remove messages you don't want to share, type words they contain (names, topics...). Every message with one of those words is removed.

`.split(",")` cuts your answer into a list at every comma, and `.strip()` removes the spaces around each word.

In [ ]:
answer = input("Type words to remove, separated by commas, or just press Enter: ")

blocked_words = []
for word in answer.lower().split(","):
    if word.strip() != "":
        blocked_words.append(word.strip())

safe_examples = []
for text in examples:
    is_safe = True
    for word in blocked_words:
        if word in text.lower():
            is_safe = False
    if is_safe:
        safe_examples.append(text)

examples = safe_examples
card(f"{len(examples)} messages will be sent to the AI", "Your clone will learn your style from them.")

if input("Do you want to see them? (y/n): ").lower() == "y":
    for text in examples:
        print("-", text)

## Step 6 · Your Style DNA

Now we measure how you text, one thing at a time. At the end we put everything in a dictionary called `style`.

**Message length and favorite laugh.** We add up the words of all your messages and divide by the number of messages. For laughs, we check if a word starts with "jaja", "haha"... using `.startswith()`.

In [ ]:
total_words = 0
for message in my_messages:
    total_words += len(message["text"].split())
average_words = round(total_words / len(my_messages), 1)

LAUGHS = ["jaja", "jeje", "jsjs", "haha", "lol", "lmao", "xd", "heh"]
laugh_counts = Counter()
for message in my_messages:
    for word in get_words(message["text"]):
        for laugh in LAUGHS:
            if word.startswith(laugh):
                laugh_counts[laugh] += 1

if laugh_counts:
    favorite_laugh = laugh_counts.most_common(1)[0][0]
else:
    favorite_laugh = None

print("Average words:", average_words)
print("Favorite laugh:", favorite_laugh)

**Signature phrase.** We look at every pair of words that go together (word 1 + word 2, word 2 + word 3...) and count which pair you repeat the most. We skip pairs where both words are boring.

In [ ]:
phrase_counts = Counter()
for message in my_messages:
    words = get_words(message["text"])
    for i in range(len(words) - 1):
        if words[i] not in STOP_WORDS or words[i + 1] not in STOP_WORDS:
            phrase_counts[words[i] + " " + words[i + 1]] += 1

signature_phrase, phrase_times = phrase_counts.most_common(1)[0]
print(f'Your signature phrase is "{signature_phrase}" ({phrase_times} times)')

**Reply speed and who starts the conversations.** If more than 60 minutes pass between two messages, we say a new conversation started. If someone else wrote and then you answered, we save how many minutes you took.

To get your usual reply time we sort the list with `.sort()` and take the one in the middle (that's called the **median**).

In [ ]:
reply_minutes = []
conversations = 1
started_by_me = 1 if messages[0]["author"] == my_name else 0

for i in range(1, len(messages)):
    previous = messages[i - 1]
    current = messages[i]
    minutes = (current["date"] - previous["date"]).total_seconds() / 60

    if minutes >= 60:
        conversations += 1
        if current["author"] == my_name:
            started_by_me += 1
    elif previous["author"] != my_name and current["author"] == my_name:
        reply_minutes.append(minutes)

reply_minutes.sort()
if len(reply_minutes) == 0:
    reply_speed = "No replies"
else:
    middle = reply_minutes[len(reply_minutes) // 2]
    if middle < 1:
        reply_speed = "Under 1 min"
    else:
        reply_speed = f"{round(middle)} min"

starts_conversations = round(100 * started_by_me / conversations)
print("Reply speed:", reply_speed)
print(f"You start {starts_conversations}% of the conversations")

**Favorite hour.** Every date has an `.hour`, so we just count them.

In [ ]:
hour_counts = Counter()
for message in my_messages:
    hour_counts[message["date"].hour] += 1

favorite_hour = hour_counts.most_common(1)[0][0]
print(f"You text the most at {favorite_hour}:00")

**Your vibe, according to the AI.** We join 80 of your example messages into one text with `"\n".join(...)` (`\n` means "new line") and ask Gemini to describe you in one sentence. This is our first **prompt**!

In [ ]:
from google.colab import ai

sample = "\n".join(random.sample(examples, min(80, len(examples))))

vibe = ai.generate_text(f"""These are WhatsApp messages written by one person:
{sample}

Describe this person's texting personality in one short, funny and honest sentence
(maximum 12 words, in English, no emojis). Only write the sentence.""").strip()

print(vibe)

Now we put everything together in the `style` dictionary and show your Style DNA.

In [ ]:
top_words = []
for word, count in word_counts.most_common(5):
    top_words.append(word)

style = {
    "vibe": vibe,
    "signature_phrase": signature_phrase,
    "phrase_times": phrase_times,
    "reply_speed": reply_speed,
    "starts_conversations": starts_conversations,
    "favorite_hour": favorite_hour,
    "average_words": average_words,
    "favorite_laugh": favorite_laugh,
    "top_words": top_words,
}

show_style_dna(style, my_name)

### Challenge 3

Because `style` is a dictionary, you can print any value with its key. Try some of these: `"vibe"`, `"signature_phrase"`, `"favorite_laugh"`, `"top_words"`.

In [ ]:
print(style["reply_speed"])

## Step 7 · Create your AI clone

An AI like Gemini doesn't know you. To make it text like you, we write a **prompt**: a long text with your Style DNA, your example messages and some rules. It's just a big f-string!

**New concept: making your own function.** With `def` we create a function called `build_prompt`. It gets the conversation so far and the new message, and returns the full prompt.

`writing_habits` measures small details (capital letters, periods...), `character_info` explains who the character is if you used the sample chat, and `history_to_text` turns the chat window's history into text.

In [ ]:
habits = writing_habits(messages, my_name)
example_lines = "\n".join(examples)
extra_rules = ""


def build_prompt(history, new_message):
    return f"""You are the AI clone of {my_name}. Reply to the friend exactly like {my_name} texts on WhatsApp.
{character_info(my_name)}

How {my_name} texts:
- Vibe: {style["vibe"]}
- About {style["average_words"]} words per message
- Favorite laugh: {style["favorite_laugh"]}
- Favorite words: {style["top_words"]}
- Signature phrase: {style["signature_phrase"]}
- Starts in lowercase in {habits["starts_lowercase"]}% of messages
- Ends with a period in {habits["ends_with_period"]}% of messages
- Uses ?? or !! in {habits["double_punctuation"]}% of messages
- Sends about {habits["messages_in_a_row"]} messages in a row (put each one on a new line)

Real messages from {my_name}:
{example_lines}

Rules:
- Reply in the same language {my_name} uses in the real messages.
- Copy the length, slang and laughs. Don't use emojis unless the real messages use them. Don't sound like an assistant.
- Only write {my_name}'s reply, nothing else.
- {extra_rules}

Conversation:
{history_to_text(history, my_name)}Friend: {new_message}
{my_name}:"""

Curious? This prints the beginning of the prompt we send to the AI. `[:1500]` means "only the first 1500 characters".

In [ ]:
print(build_prompt([], "hey what are you doing?")[:1500])

Now the function that talks to the AI: it builds the prompt and sends it with `ai.generate_text()`. We'll use `clone_reply` for the chat and for the game.

In [ ]:
def clone_reply(new_message, history=[]):
    prompt = build_prompt(history, new_message)
    return ai.generate_text(prompt).strip()


print(clone_reply("hey what are you doing?"))

### Challenge 4

Give your clone a secret rule! In the `build_prompt` cell, write something inside `extra_rules`, like `"Always mention pizza"` or `"Answer like a pirate"`. Run that cell and the `clone_reply` cell again, and see how the answer changes. (Set it back to `""` before the game!)

## Step 8 · Chat with your clone

Run this cell and open the link that ends in `.gradio.live`. You can send it to your friends: it works for 72 hours, while this notebook is open.

In [ ]:
start_chat(clone_reply, my_name)

## Step 9 · Real or Clone?

Time to test your clone. For each of the 5 real moments we saved in Step 5, your clone answers the same message. Then we mix its answer with your real one using `random.shuffle()`, so nobody knows which one comes first.

We keep only the first line of the clone's answer with `.split("\n")[0]`. It takes about a minute, because the AI writes 5 answers.

In [ ]:
rounds = []

for friend_message, real_reply in game_moments:
    clone_answer = clone_reply(friend_message).split("\n")[0]
    options = [real_reply, clone_answer]
    random.shuffle(options)
    rounds.append({"friend": friend_message, "real": real_reply, "options": options})
    print(f"Round {len(rounds)} ready")

Now play! Open the `.gradio.live` link and give it to the person next to you (or send it to a friend). Can they tell which reply is really yours?

In [ ]:
start_game(rounds, my_name)

## What you learned today

Variables, functions, lists, dictionaries, `for` loops, `if` conditions, f-strings, counting with `Counter`, and how to talk to an AI with a prompt. That's real Python!

Want to keep going? Come to the next IE Coding Club workshop.